# DiffSynth 원본 Anima 추론 · Colab T4

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고 위에서부터 실행합니다.
공식 DiffSynth-Studio의 고정 커밋을 독립 가상환경에 설치하고 `AnimaImagePipeline`을 직접 호출합니다.

- Python 3.13.15, PyTorch 2.14.0+cu130, torchvision 0.29.0+cu130을 사용합니다.
- 원본 tokenizer·Qwen·DiT·VAE·FlowMatch Euler를 사용합니다. ComfyUI 함수 및 기존 프로젝트 실행 코드는 불러오지 않습니다.
- T4용 FP16과 CPU offload는 DiffSynth의 공개 로딩 옵션으로 지정합니다.
- 원본 Z-Image 시간표와 원본 난수 생성을 사용하므로 기존 `matched_euler`와 계산 조건이 다릅니다.
- 원본 추론의 T4 성공 여부와 이미지 품질은 실제 실행 결과로 확인합니다.

[공식 Anima 사용법](https://github.com/modelscope/DiffSynth-Studio/blob/7686e54d41d25c0e8ed5f1318acc23b6bb832654/docs/en/Model_Details/Anima.md)

## 1. 실행 설정

기존 이미지의 프롬프트·seed·크기·steps·CFG를 초기값으로 사용합니다. 아래 사전에서 변경할 수 있습니다.
`generation`에는 원본 API가 받는 인자만 전달합니다. `sigma_shift=3.0`도 원본 API 옵션이며 시간표 배열은 교체하지 않습니다.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

WORK = Path('/content/diffsynth_original')
SOURCE = WORK / 'DiffSynth-Studio'
VENV = WORK / '.venv'
PYTHON = VENV / 'bin' / 'python'
WORK.mkdir(parents=True, exist_ok=True)

SETTINGS = {'repository': 'https://github.com/modelscope/DiffSynth-Studio.git',
 'revision': '7686e54d41d25c0e8ed5f1318acc23b6bb832654',
 'environment': {'python': '3.13.15',
                 'torch': '2.14.0+cu130',
                 'torchvision': '0.29.0+cu130',
                 'torch_index': 'https://download.pytorch.org/whl/cu130',
                 'uv': '0.12.18',
                 'transformers': '4.57.6',
                 'accelerate': '1.12.0',
                 'peft': '0.18.1',
                 'huggingface-hub': '0.36.0'},
 'model': {'repository': 'circlestone-labs/Anima',
           'revision': 'f973fc41ec7545364ac9776c2440285f43ff2a30',
           'files': ['split_files/diffusion_models/anima-base-v1.0.safetensors',
                     'split_files/text_encoders/qwen_3_06b_base.safetensors',
                     'split_files/vae/qwen_image_vae.safetensors']},
 'tokenizers': {'qwen': {'repository': 'Qwen/Qwen3-0.6B', 'revision': 'main', 'subfolder': ''},
                't5': {'repository': 'stabilityai/stable-diffusion-3.5-large',
                       'revision': 'main',
                       'subfolder': 'tokenizer_3'}},
 'dtype': 'float16',
 'vram_margin_gib': 1.5,
 'generation': {'prompt': '@ogipote, 1girl, plana (blue archive), upper body, \n'
                          ', solo, cute, cheerful, innocent, expressionless, looking_at_viewer, '
                          'alternte costume, white school uniform, blue_ribbon, white_socks, '
                          'school_theme, sitting, holding_tablet, sunlight, soft_lighting, '
                          'gentle_breeze, floating_particles, futuristic_city, academy_city, '
                          'glass_building, blue_and_white_theme, pastel_colors, vibrant_colors, '
                          'depth_of_field, wallpaper',
                'negative_prompt': 'worst quality, low quality, score_1, score_2, score_3, blurry, '
                                   'jpeg artifacts',
                'seed': 1113280783077040,
                'height': 832,
                'width': 1216,
                'num_inference_steps': 30,
                'cfg_scale': 4.0,
                'denoising_strength': 1.0,
                'sigma_shift': 3.0,
                'rand_device': 'cpu'}}

CONFIG = WORK / 'settings.json'
CONFIG.write_text(json.dumps(SETTINGS, ensure_ascii=False, indent=2), encoding='utf-8')

# 노트북 커널과 독립된 Python 프로세스를 사용합니다.
def run_command(command, env=None):
    with subprocess.Popen([str(x) for x in command], cwd=WORK, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'실행 실패: 종료 코드 {process.returncode}')

print('작업 폴더:', WORK)

## 2. 공식 소스와 독립 환경 설치

Colab 기본 패키지를 상속하지 않는 가상환경을 사용합니다. torch와 torchvision 버전은 의존성 설치 중에도 고정합니다.
Transformers 관련 버전은 기존 DiffSynth 환경과 같은 호환 조합입니다. Anima 이미지 추론에 필요하지 않은 torchaudio는 설치하지 않습니다.

In [ ]:
import hashlib
import shutil
import urllib.request
import zipfile

e = SETTINGS['environment']
uv = shutil.which('uv')
if uv is None:
    with urllib.request.urlopen(f"https://pypi.org/pypi/uv/{e['uv']}/json") as response:
        wheels = json.load(response)['urls']
    wheel_info = next(w for w in wheels if w['filename'].endswith('.whl')
                      and 'manylinux_2_17_x86_64' in w['filename'])
    wheel = WORK / wheel_info['filename']
    urllib.request.urlretrieve(wheel_info['url'], wheel)
    if hashlib.sha256(wheel.read_bytes()).hexdigest() != wheel_info['digests']['sha256']:
        raise RuntimeError('uv 다운로드 해시가 일치하지 않습니다.')
    uv = WORK / 'uv'
    with zipfile.ZipFile(wheel) as archive:
        member = next(n for n in archive.namelist() if n.endswith('/uv'))
        uv.write_bytes(archive.read(member))
    uv.chmod(0o755)

if not SOURCE.exists():
    run_command(['git', 'clone', '--no-checkout', SETTINGS['repository'], SOURCE])
run_command(['git', '-C', SOURCE, 'checkout', '--detach', SETTINGS['revision']])
run_command([uv, 'venv', '--python', e['python'], VENV]) if not PYTHON.exists() else None
constraints = WORK / 'constraints.txt'
constraints.write_text(''.join(f'{name}=={e[name]}\n' for name in
    ['torch', 'torchvision', 'transformers', 'accelerate', 'peft', 'huggingface-hub']), encoding='utf-8')
run_command([uv, 'pip', 'install', '--python', PYTHON,
             f"torch=={e['torch']}", f"torchvision=={e['torchvision']}",
             '--index-url', e['torch_index']])
run_command([uv, 'pip', 'install', '--python', PYTHON, '-e', SOURCE,
             'sentencepiece', 'protobuf', 'huggingface-hub',
             '-c', constraints, '--extra-index-url', e['torch_index']])
run_command([PYTHON, '-c', "import sys, torch, torchvision; print('Python:', sys.version); print('torch:', torch.__version__); print('torchvision:', torchvision.__version__); print('CUDA build:', torch.version.cuda); print('GPU:', torch.cuda.get_device_name(0))"])

## 3. 모델과 원본 토크나이저 다운로드

원본 Anima 예제의 Qwen·T5 토크나이저 저장소를 사용합니다. 토크나이저는 실행 시 확인한 revision을 기록합니다.
T5 저장소 접근에 인증이 필요한 경우 Hugging Face에서 해당 저장소의 이용 승인을 받은 뒤 Colab 보안 비밀에 `HF_TOKEN`을 등록합니다.

In [ ]:
DOWNLOAD_CODE = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

config = Path(sys.argv[1])
cfg = json.loads(config.read_text())
work = config.parent
model = cfg['model']
paths = [hf_hub_download(repo_id=model['repository'], revision=model['revision'], filename=name)
         for name in model['files']]
manifest = {'model': model, 'weight_paths': paths, 'tokenizers': {}}
for name, item in cfg['tokenizers'].items():
    revision = HfApi().model_info(item['repository'], revision=item['revision']).sha
    prefix = item['subfolder'].strip('/')
    patterns = [f'{prefix}/*'] if prefix else ['*.json', '*.txt', '*.model', '*.jinja']
    folder = snapshot_download(repo_id=item['repository'], revision=revision, allow_patterns=patterns)
    manifest['tokenizers'][name] = dict(item, resolved_revision=revision,
                                       path=str(Path(folder) / prefix))
(work / 'downloads.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('가중치 3개와 토크나이저 준비 완료')
"""
child_env = os.environ.copy()
try:
    from google.colab import userdata
    if not child_env.get('HF_TOKEN'):
        try:
            child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
except ImportError:
    pass
run_command([PYTHON, '-u', '-c', DOWNLOAD_CODE, CONFIG], env=child_env)

## 4. 원본 파이프라인으로 생성

`AnimaImagePipeline.from_pretrained(...)`로 로딩한 뒤 `pipe(**generation)`을 직접 호출합니다.
초기 노이즈·시간표·CFG·텍스트 처리·DiT·VAE는 원본 구현이 담당합니다.
이미지와 실행 설정, 실제 환경, 원본 시간표를 실행별 폴더에 저장합니다.

In [ ]:
INFERENCE_CODE = r"""
import json, sys, time, platform, subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
import torch
from diffsynth.pipelines.anima_image import AnimaImagePipeline, ModelConfig

config = Path(sys.argv[1])
cfg = json.loads(config.read_text())
work = config.parent
assets = json.loads((work / 'downloads.json').read_text())
dtype = getattr(torch, cfg['dtype'])
output = work / 'results' / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
output.mkdir(parents=True, exist_ok=False)
(output / 'settings.json').write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
(output / 'downloads.json').write_text(json.dumps(assets, ensure_ascii=False, indent=2))
environment = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'cuda_build': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
    'packages': {name: version(name) for name in ['torchvision', 'transformers', 'accelerate', 'peft', 'huggingface-hub']},
    'source_commit': subprocess.check_output(['git', '-C', str(work / 'DiffSynth-Studio'), 'rev-parse', 'HEAD'], text=True).strip(),
}
(output / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, ensure_ascii=False, indent=2))
vram = dict(offload_dtype=dtype, offload_device='cpu', onload_dtype=dtype, onload_device='cpu',
            preparing_dtype=dtype, preparing_device='cuda', computation_dtype=dtype, computation_device='cuda')
pipe = AnimaImagePipeline.from_pretrained(
    torch_dtype=dtype, device='cuda',
    model_configs=[ModelConfig(path=path, **vram) for path in assets['weight_paths']],
    tokenizer_config=ModelConfig(path=assets['tokenizers']['qwen']['path']),
    tokenizer_t5xxl_config=ModelConfig(path=assets['tokenizers']['t5']['path']),
    vram_limit=torch.cuda.mem_get_info()[0] / 2**30 - cfg['vram_margin_gib'],
)
torch.cuda.synchronize()
started = time.perf_counter()
image = pipe(**cfg['generation'])
torch.cuda.synchronize()
seconds = time.perf_counter() - started
image.save(output / 'image.png')
(output / 'schedule.json').write_text(json.dumps({
    'class': type(pipe.scheduler).__name__,
    'sigmas': pipe.scheduler.sigmas.tolist(),
    'timesteps': pipe.scheduler.timesteps.tolist(),
}, indent=2))
(output / 'result.json').write_text(json.dumps({'generation_seconds': seconds, 'image_size': list(image.size)}, indent=2))
(work / 'last_result.txt').write_text(str(output))
print('생성 시간:', round(seconds, 2), '초')
print('결과:', output)
"""
run_command([PYTHON, '-u', '-c', INFERENCE_CODE, CONFIG], env=child_env)

## 5. 결과 확인과 다운로드

아래 ZIP에 이미지와 설정·환경·시간표가 포함됩니다.

In [ ]:
from IPython.display import Image, display

RESULT = Path((WORK / 'last_result.txt').read_text().strip())
display(Image(filename=str(RESULT / 'image.png')))
archive = shutil.make_archive(str(RESULT), 'zip', root_dir=RESULT)
print('결과 ZIP:', archive)
from google.colab import files
files.download(archive)